# Phase 3A — Dataset, label, corrected manifest QA

**목적:** natural validation, task-local quota, leakage QA, overlap payload hash가 포함된 corrected manifest를 검증한다.  
**입력:** Phase 2 natural/target datasets와 fixed episode split.  
**출력:** `phase3B_R1_eval_manifest_v2.json`.  
기존 v1 manifest와 legacy 36-run 결과는 변경하지 않는다.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys, subprocess
root = Path(os.environ.get('GRAPH_CLAD_PROJECT_ROOT', Path.cwd())).resolve()
if not (root / 'scripts').is_dir() and Path('/content/Graph-CLaD').is_dir(): root = Path('/content/Graph-CLaD')
os.chdir(root); sys.path.insert(0, str(root)) if str(root) not in sys.path else None
from scripts.research_paths import resolve_research_paths
paths = resolve_research_paths(project_root=root)
corrected_root = paths.artifact_root / 'phase3_holder_action_v1' / 'corrected_protocol_v2'
manifest_path = corrected_root / 'phase3B_R1_eval_manifest_v2.json'
eval_config = paths.config_root / 'phase3_holder_action_eval_v2_corrected.json'

## Fixed protocol
Checkpoint: natural-validation conditional event PR-AUC. Threshold: natural validation에서 선택 후 모든 test/stress/control에 고정. Natural test가 primary이고 challenge는 stress subset이다.

In [ ]:
RUN_MANIFEST_BUILD = False
manifest_cmd = [sys.executable, '-m', 'scripts.phase3.build_eval_manifest',
                '--config', str(eval_config), '--output', str(manifest_path)]
print(' '.join(manifest_cmd))
if RUN_MANIFEST_BUILD:
    if manifest_path.exists():
        raise FileExistsError('Use a new manifest version; refusing to overwrite existing manifest')
    subprocess.run(manifest_cmd, check=True)

In [ ]:
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
summary = {
    'status': manifest.get('status'),
    'sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'relations': manifest.get('relations'),
    'folds': [f.get('name') for f in manifest.get('folds', [])],
    'validation_protocol': manifest.get('validation_protocol'),
}
summary

In [ ]:
assert manifest.get('status') == 'pass'
assert manifest.get('validation_protocol', {}).get('source') == 'natural'
for fold in manifest.get('folds', []):
    overlap = fold.get('overlap_payload_hash_qa', {})
    assert overlap.get('challenge_subset_of_natural')
    assert overlap.get('payload_hash_mismatch_count') == 0
{'qa': 'pass', 'next': 'phase_3b_corrected_architecture_gate.ipynb'}

## 생성물, 검증, 다음 phase
Manifest SHA를 실행 config/runtime manifest와 함께 보존한다. Sample/episode leakage, task-local quota, overlap payload hash가 pass해야 학습으로 이동한다. Weak-label 수동 audit은 자동 판정으로 대체하지 않는다. 다음은 `phase_3b_corrected_architecture_gate.ipynb`다.